In [1]:
import pandas as pd
import numpy as np
import requests
from src.data_loader import _load_single_gsheet, _get_gsheet_client, _load_single_gsheet_ci
from src import config

2026-03-24 14:34:16.403 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


In [2]:
#df_oinm_raw = pd.read_excel(io="data/OINM.xlsx")
gsheet_client = _get_gsheet_client() 
#df_oinm_raw = _load_single_gsheet(gsheet_client, config.GSHEET_IDS["OINM"], "OINM")
df_stock_raw = pd.read_excel(io="data/Stock.xlsx")
df_ci_raw = _load_single_gsheet_ci(gsheet_client, sheet_id= config.GSHEET_IDS["CI"], sheet_name= "Pipedrive Deals")

2026-03-24 14:34:16.420 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-24 14:34:17.337 
'server.enableXsrfProtection=true'.
As a result, 'server.enableCORS' is being overridden to 'true'.

More information:
In order to protect against CSRF attacks, we send a cookie with each request.
To do so, we must specify allowable origins, which places a restriction on
cross-origin resource sharing.

If cross origin resource sharing is required, please disable server.enableXsrfProtection.
            
2026-03-24 14:34:17.340 
  command:

    streamlit run C:\Users\m.fernandez_fluxsola\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-03-24 14:34:17.341 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-

In [3]:
df_ci_raw

,Título,ID,CeCo,Licitación,Grupo Licitación,Estado,Stage name,Fecha de la última actividad,Fecha de ganado,Fecha de perdido,...,50kWac,100kWac,150kWac,Enviados Equipos?,Modelo vacio,Panel,Estructuras,Inversores,Filtro 3 posteriores meses,Considerar?
0,Mall Plaza de Los Rios - 18018,18018,,No,no,open,Viabilidad Técnica,,,,...,0,0,0,No,Faltante,Sin panel,,Sin inversores,FALSE,
1,Club de Golf Lomas de la Dehesa - 18195,18195,,No,N/A,open,Propuesta definitiva entregada,,,,...,0,0,1,No,Sin faltante,,,,FALSE,
2,Templo Votivo Maipú - 16833,16833,,No,N/A,open,Propuesta definitiva entregada,,,,...,0,1,0,No,Sin faltante,,,,TRUE,
3,Luminosos Alcaino - 13429,13429,,No,N/A,open,Viabilidad Técnica,,,,...,0,2,0,No,Sin faltante,,,,TRUE,
4,Ponhuipa II - 15617,15617,CVDI-98,No,na,won,Negociación Contrato,,2026-02-27 0:00:00,,...,0,0,2,No,Sin faltante,,,,FALSE,Orgánico
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2994,,,,,,,,,,,...,0,0,0,No,Faltante,Sin panel,Sin estructuras,Sin inversores,FALSE,
2995,,,,,,,,,,,...,0,0,0,No,Faltante,Sin panel,Sin estructuras,Sin inversores,FALSE,
2996,,,,,,,,,,,...,0,0,0,No,Faltante,Sin panel,Sin estructuras,Sin inversores,FALSE,
2997,,,,,,,,,,,...,0,0,0,No,Faltante,Sin panel,Sin estructuras,Sin inversores,FALSE,


In [4]:
df_oinm_clean = df_oinm_raw.copy()

In [5]:
# 1. Convertimos la columna a texto y reemplazamos TODOS los guiones por barras
fechas_unificadas = df_oinm_clean['fecha'].astype(str).str.replace('-', '/')

# 2. Ahora sí, convertimos a fecha. Pandas verá todo uniforme como DD/MM/YYYY
df_oinm_clean['fecha'] = pd.to_datetime(
    fechas_unificadas, 
    dayfirst=True, 
    errors='coerce'
)

cols_numericas_complejas = [
        'CantidadMovimiento', 
        'ValorMovimiento', 
        'CantidadSalida'
    ]
    
for col in cols_numericas_complejas:
        if col in df_oinm_clean.columns:
            # Protección contra datos ya numéricos
            if pd.api.types.is_numeric_dtype(df_oinm_clean[col]):
                continue
            # A. Convertir a texto y quitar espacios
            serie_limpia = df_oinm_clean[col].astype(str).str.strip()
            
            # B. Quitar el separador de miles (ej. "8.642,00" -> "8642,00")
            serie_limpia = serie_limpia.str.replace('.', '', regex=False)
            
            # C. Cambiar la coma decimal por punto (ej. "8642,00" -> "8642.00")
            serie_limpia = serie_limpia.str.replace(',', '.', regex=False)
            
            # D. Forzar a número
            df_oinm_clean[col] = pd.to_numeric(serie_limpia, errors='coerce')
            
            # E. Rellenar vacíos con 0
            df_oinm_clean[col] = df_oinm_clean[col].fillna(0)

In [6]:
df_oinm_clean

,fecha,CodigoArticulo,NombreArticulo,Familia,SubFamilia,CodigoBodega,CantidadMovimiento,ValorMovimiento
0,2023-12-31,EXI-003968,Conector Recto 25Mm De Conduit A Flexible Ekol...,MATERIALES,,BF0001,3285.0,3636495.0
1,2023-12-31,EXI-003901,"Int. Automatico 2X25A C 6/10Ka, Legrand",MATERIALES,,BF0001,541.0,9725557.0
2,2023-12-31,EXI-000635,"Cable Rz1-K 16 Mm 0.6/1Kv, Negro",MATERIALES,,BF0001,18045.0,32011830.0
3,2023-12-31,EXI-003628,Cable Rv-K Negro 4/0 Awg,MATERIALES,,BF0001,1027.0,10351133.0
4,2023-12-31,EXI-001252,Cable Rz1-K 10 Mm Negro,MATERIALES,,BF0001,13958.0,16414608.0
...,...,...,...,...,...,...,...,...
79408,2026-03-24,EXI-001298,Tornillo Autop. C/Gol 12-14 X 3,MATERIALES,,BF0001,-100.0,-3956.0
79409,2026-03-24,EXI-001297,Tornillo Autop. C/Gol. 12-14 X 2,MATERIALES,,BF0001,-100.0,-2454.0
79410,2026-03-24,EXI-001294,Tornillo Autop. C/Gol. 12-14 X 1,MATERIALES,,BF0001,-100.0,-1840.0
79411,2026-03-24,EXI-000642,Tornillo Autop #3 C/Gol. 12-14 X 1.1/2,MATERIALES,,BF0001,-100.0,-2366.0


In [7]:
df_mensual = df_oinm_clean.resample('YS', on='fecha')['ValorMovimiento'].sum().reset_index().sort_values('fecha')
df_mensual

,fecha,ValorMovimiento
0,2023-01-01,8.944613e+09
1,2024-01-01,-3.600466e+09
2,2025-01-01,-1.013954e+09
3,2026-01-01,-6.927295e+07


In [8]:
df_mensual["ValorAcumulado"] = df_mensual["ValorMovimiento"].cumsum()

In [13]:
df_oinm_clean.resample('YS', on='fecha')['ValorMovimiento'].sum()

fecha
2023-01-01    8.944613e+09
2024-01-01   -3.600466e+09
2025-01-01   -1.013954e+09
2026-01-01   -6.927295e+07
Freq: YS-JAN, Name: ValorMovimiento, dtype: float64